# 04 — Geometry Optimization: Finding Energy Minima

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ppt-2/Ch121a-DFT/blob/main/notebooks/04_geometry_optimization.ipynb)

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:
- Understand potential energy surfaces (PES) and what geometry optimization means
- Use PySCF with geomeTRIC to optimize molecular geometries
- Compare optimized geometries with experimental values
- Understand convergence criteria and common pitfalls in geometry optimization
- Write ORCA input files for geometry optimization
- Analyze how the method/basis affects optimized geometries

## 1. Theory: Potential Energy Surfaces and Optimization

### 1.1 The Born-Oppenheimer Potential Energy Surface

Within the Born-Oppenheimer approximation, electrons adjust instantaneously to
nuclear motion. The electronic energy as a function of nuclear coordinates
$\mathbf{R} = \{R_I\}$ defines the **potential energy surface (PES)**:

$$E_{BO}(\mathbf{R}) = \langle \Psi_{elec} | \hat{H}_{elec} | \Psi_{elec} \rangle$$

**Stationary points** on the PES satisfy:
$$\mathbf{g}(\mathbf{R}) = \frac{\partial E}{\partial \mathbf{R}} = \mathbf{0}$$

- **Minimum**: all eigenvalues of the Hessian matrix $\mathbf{H} = \frac{\partial^2 E}{\partial R_i \partial R_j}$ are positive
- **Transition state (TS)**: exactly one negative Hessian eigenvalue (imaginary frequency)
- **Higher-order saddle point**: two or more negative eigenvalues

### 1.2 Gradient-Based Optimization Algorithms

Modern geometry optimizers compute the energy gradient $\mathbf{g}$ analytically
and use it to step toward the minimum:

**Steepest Descent**: $\mathbf{R}_{n+1} = \mathbf{R}_n - \alpha \mathbf{g}_n$
(simple but slow convergence)

**Quasi-Newton (BFGS/L-BFGS)**:
$$\mathbf{R}_{n+1} = \mathbf{R}_n - \mathbf{H}_n^{-1} \mathbf{g}_n$$
where the approximate inverse Hessian $\mathbf{H}_n^{-1}$ is updated at each step.
This is the algorithm used in most QC codes (e.g., geomeTRIC, DL-FIND, Berny).

**RFO (Rational Function Optimization)**: Finds both minima and transition states
by solving a modified Newton equation with level-shift parameter.

### 1.3 Convergence Criteria

A geometry optimization is typically considered converged when ALL of these are satisfied:

| Criterion | Tight threshold | Default threshold |
|-----------|:--------------:|:-----------------:|
| Max gradient component | $< 1.5 \times 10^{-5}$ Ha/\u00c5 | $< 4.5 \times 10^{-4}$ Ha/\u00c5 |
| RMS gradient | $< 1.0 \times 10^{-5}$ Ha/\u00c5 | $< 3.0 \times 10^{-4}$ Ha/\u00c5 |
| Max displacement | $< 6.0 \times 10^{-5}$ \u00c5 | $< 1.8 \times 10^{-3}$ \u00c5 |
| RMS displacement | $< 4.0 \times 10^{-5}$ \u00c5 | $< 1.2 \times 10^{-3}$ \u00c5 |

### 1.4 Internal vs Cartesian Coordinates

Cartesian coordinates: $3N$ degrees of freedom (including 6 translations/rotations)
Internal coordinates (Z-matrix): bond lengths, bond angles, dihedral angles
**Delocalized internal coordinates** (geomeTRIC): linear combinations of primitive
internals \u2014 generally the fastest convergence.

In [ ]:
# =============================================================================
# Ch121a: Quantum Chemistry & DFT — Notebook 04: Geometry Optimization
# License: GPL-3.0 (https://www.gnu.org/licenses/gpl-3.0.en.html)
# =============================================================================
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyscf import gto, dft, scf

# ------------------------------------------------------------------
# Morse Potential: Realistic Bond Potential Energy Surface
# ------------------------------------------------------------------
# The Morse potential is a good approximation to a diatomic bond:
#   V(r) = D_e * [1 - exp(-α(r - r_e))]^2
#
# Parameters for H-F bond:
De = 5.869    # eV — dissociation energy from minimum
alpha = 2.287 # Å^-1
r_e = 0.917   # Å — equilibrium bond length

r = np.linspace(0.5, 4.5, 500)  # Angstrom

# Morse potential
V_morse = De * (1 - np.exp(-alpha * (r - r_e)))**2

# Harmonic approximation: V = (1/2) k (r - r_e)^2
# Force constant k = 2 * D_e * alpha^2 (eV/Å^2)
k = 2 * De * alpha**2
V_harmonic = 0.5 * k * (r - r_e)**2

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(r, V_morse, '-', color='steelblue', linewidth=2.5, label='Morse potential')
ax.plot(r, V_harmonic, '--', color='coral', linewidth=2, label='Harmonic approximation')
ax.axhline(y=De, color='seagreen', linestyle=':', linewidth=1.5, alpha=0.8,
           label=f'Dissociation limit $D_e$ = {De:.3f} eV')
ax.axvline(x=r_e, color='gray', linestyle=':', linewidth=1.2, alpha=0.7,
           label=f'Equilibrium $r_e$ = {r_e:.3f} Å')

# Annotate
ax.annotate('Equilibrium\nminimum', xy=(r_e, 0), xytext=(r_e+0.4, 0.5),
            arrowprops=dict(arrowstyle='->', color='gray'),
            fontsize=9, ha='center')
ax.annotate('Dissociation limit', xy=(4.0, De), xytext=(3.0, De+0.5),
            arrowprops=dict(arrowstyle='->', color='seagreen'),
            fontsize=9, color='seagreen')

# Shade anharmonic region
ax.fill_between(r, V_harmonic, V_morse,
                where=(r > r_e + 0.3) & (V_harmonic < De),
                alpha=0.15, color='coral', label='Anharmonicity')

ax.set_xlim(0.5, 4.5)
ax.set_ylim(-0.3, De + 1.5)
ax.set_xlabel('H-F Bond Distance (Å)', fontsize=12)
ax.set_ylabel('Potential Energy (eV)', fontsize=12)
ax.set_title('Morse Potential for H-F Bond\nvs Harmonic Approximation', fontsize=12)
ax.legend(fontsize=9, loc='upper right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'H-F bond parameters:')
print(f'  Equilibrium distance r_e = {r_e:.3f} Å')
print(f'  Dissociation energy  D_e = {De:.3f} eV = {De*23.06:.1f} kcal/mol')
print(f'  Force constant       k   = {k:.2f} eV/Å² = {k*1.6022e-19/1e-20:.0f} N/m')

In [ ]:
%%time
# ------------------------------------------------------------------
# Geometry Optimization of Water: B3LYP/def2-SVP
# ------------------------------------------------------------------
# We use PySCF's geometric_solver (geomeTRIC optimizer)
# Starting from a slightly distorted geometry to test convergence.

from pyscf import gto, dft
from pyscf.geomopt import geometric_solver
import numpy as np
from pyscf.data.nist import BOHR  # Bohr to Angstrom conversion

# Slightly distorted initial geometry
mol = gto.Mole()
mol.atom = '''
O   0.000000   0.000000   0.100000
H   0.000000   0.800000  -0.400000
H   0.000000  -0.800000  -0.400000
'''
mol.basis = 'def2-SVP'
mol.verbose = 0
mol.build()

mf = dft.RKS(mol)
mf.xc = 'B3LYP'
mf.verbose = 0

print('Starting geometry (distorted):')
print(f"  {'Atom':5s}  {'x (Å)':>10s}  {'y (Å)':>10s}  {'z (Å)':>10s}")
for i in range(mol.natm):
    sym = mol.atom_symbol(i)
    coords = mol.atom_coord(i) * BOHR  # Convert from Bohr to Angstrom
    print(f"  {sym:5s}  {coords[0]:10.6f}  {coords[1]:10.6f}  {coords[2]:10.6f}")

# Run geometry optimization
conv_mol = geometric_solver.optimize(mf, verbose=0)

print('\nOptimized geometry (B3LYP/def2-SVP):')
print(f"  {'Atom':5s}  {'x (Å)':>10s}  {'y (Å)':>10s}  {'z (Å)':>10s}")
final_coords = []
for i in range(conv_mol.natm):
    sym = conv_mol.atom_symbol(i)
    coords = conv_mol.atom_coord(i) * BOHR
    final_coords.append(coords)
    print(f"  {sym:5s}  {coords[0]:10.6f}  {coords[1]:10.6f}  {coords[2]:10.6f}")

# Compute bond lengths and angle
O_pos = final_coords[0]
H1_pos = final_coords[1]
H2_pos = final_coords[2]

r_OH1 = np.linalg.norm(H1_pos - O_pos)
r_OH2 = np.linalg.norm(H2_pos - O_pos)
# Bond angle
v1 = H1_pos - O_pos
v2 = H2_pos - O_pos
cos_angle = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
angle = np.degrees(np.arccos(cos_angle))

print(f'\nOptimized structural parameters:')
print(f'  r(O-H₁) = {r_OH1:.4f} Å  (exp: 0.9572 Å)')
print(f'  r(O-H₂) = {r_OH2:.4f} Å  (exp: 0.9572 Å)')
print(f'  ∠(H-O-H) = {angle:.2f}°  (exp: 104.52°)')
print(f'\n  Error r(OH): {abs(r_OH1-0.9572)*1000:.1f} mÅ')
print(f'  Error angle: {abs(angle-104.52):.2f}°')

In [ ]:
%%time
# ------------------------------------------------------------------
# Systematic Comparison: B3LYP/def2-SVP vs Experiment
# ------------------------------------------------------------------

from pyscf import gto, dft
from pyscf.geomopt import geometric_solver
from pyscf.data.nist import BOHR
import numpy as np
import pandas as pd

# Experimental reference data
# Format: name, atom_str, bond_label, bond_idx_pair, exp_value (Å), angle_label, angle_idx, exp_angle (deg)
molecules = [
    {
        'name': 'H₂O',
        'atom': 'O 0 0 0.117; H 0 0.757 -0.469; H 0 -0.757 -0.469',
        'bond_pair': (0, 1),
        'bond_label': 'r(O-H)',
        'exp_bond': 0.9572,
        'angle_triplet': (1, 0, 2),
        'angle_label': '∠(H-O-H)',
        'exp_angle': 104.52,
    },
    {
        'name': 'HF',
        'atom': 'H 0 0 0; F 0 0 0.917',
        'bond_pair': (0, 1),
        'bond_label': 'r(H-F)',
        'exp_bond': 0.9168,
        'angle_triplet': None,
        'angle_label': None,
        'exp_angle': None,
    },
    {
        'name': 'CO',
        'atom': 'C 0 0 0; O 0 0 1.128',
        'bond_pair': (0, 1),
        'bond_label': 'r(C-O)',
        'exp_bond': 1.1283,
        'angle_triplet': None,
        'angle_label': None,
        'exp_angle': None,
    },
    {
        'name': 'NH₃',
        'atom': 'N 0 0 0.116; H 0 0.939 -0.269; H 0.814 -0.469 -0.269; H -0.814 -0.469 -0.269',
        'bond_pair': (0, 1),
        'bond_label': 'r(N-H)',
        'exp_bond': 1.012,
        'angle_triplet': (1, 0, 2),
        'angle_label': '∠(H-N-H)',
        'exp_angle': 106.67,
    },
]

results = []
for mol_data in molecules:
    mol = gto.Mole()
    mol.atom = mol_data['atom']
    mol.basis = 'def2-SVP'
    mol.verbose = 0
    mol.build()
    
    mf = dft.RKS(mol)
    mf.xc = 'B3LYP'
    mf.verbose = 0
    
    conv_mol = geometric_solver.optimize(mf, verbose=0)
    
    coords = np.array([conv_mol.atom_coord(i) * BOHR for i in range(conv_mol.natm)])
    i, j = mol_data['bond_pair']
    bond_len = np.linalg.norm(coords[j] - coords[i])
    
    row = {
        'Molecule': mol_data['name'],
        'Parameter': mol_data['bond_label'],
        'Computed (Å)': round(bond_len, 4),
        'Experiment (Å)': mol_data['exp_bond'],
        'Error (mÅ)': round((bond_len - mol_data['exp_bond'])*1000, 1),
    }
    
    if mol_data['angle_triplet']:
        a, b, c = mol_data['angle_triplet']
        v1 = coords[a] - coords[b]
        v2 = coords[c] - coords[b]
        cos_a = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))
        angle = np.degrees(np.arccos(np.clip(cos_a, -1, 1)))
        row['Angle Param'] = mol_data['angle_label']
        row['Computed (°)'] = round(angle, 2)
        row['Exp (°)'] = mol_data['exp_angle']
    else:
        row['Angle Param'] = '—'
        row['Computed (°)'] = '—'
        row['Exp (°)'] = '—'
    
    results.append(row)
    print(f"  {mol_data['name']:5s}: {mol_data['bond_label']} = {bond_len:.4f} Å  "
          f"(exp: {mol_data['exp_bond']:.4f} Å, Δ = {(bond_len-mol_data['exp_bond'])*1000:.1f} mÅ)")

df_geom = pd.DataFrame(results)
print('\n')
print('Geometry Comparison: B3LYP/def2-SVP vs Experiment')
print(df_geom[['Molecule','Parameter','Computed (Å)','Experiment (Å)','Error (mÅ)']].to_string(index=False))
mae_bond = np.mean(np.abs([r['Error (mÅ)'] for r in results]))
print(f'\nMean absolute error in bond lengths: {mae_bond:.1f} mÅ')

## 2. ORCA Geometry Optimization Input

The same B3LYP/def2-SVP geometry optimization in ORCA:

```
# ORCA geometry optimization of water
# Method: B3LYP-D3(BJ)/def2-SVP
! B3LYP-D3BJ def2-SVP def2/J RIJCOSX Opt

%pal
  nprocs 4    # Number of parallel cores
end

%geom
  MaxIter  100         # Maximum optimization steps
  Convergence tight    # Tight convergence criteria
  MaxStep  0.3         # Maximum step size in internal coordinates
end

* xyz 0 1
  O   0.000000   0.000000   0.100000
  H   0.000000   0.800000  -0.400000
  H   0.000000  -0.800000  -0.400000
*
```

**Common ORCA optimization keywords:**
- `Opt` — standard geometry optimization (minimize)
- `OptTS` — transition state optimization (maximize along one mode)
- `Convergence loose/normal/tight/verytight` — convergence criteria
- `def2/J` with `RIJCOSX` — density fitting for faster Coulomb integrals
- `D3BJ` — Becke-Johnson damped Grimme D3 dispersion

## 3. Common Pitfalls in Geometry Optimization

### ⚠️ 1. Starting Geometry Too Distorted
Optimization may converge to a **local minimum** if the initial geometry is far from
the global minimum. Always use chemically reasonable starting coordinates.

### ⚠️ 2. Wrong Spin State
For open-shell molecules (radicals, transition metal complexes), make sure `mol.spin`
is set correctly. A wrong spin state gives a different PES.
- `mol.spin = 0` → singlet (all electrons paired)
- `mol.spin = 2` → triplet (two unpaired electrons, $M_S = 1$)

### ⚠️ 3. Basis Set Too Small
STO-3G geometries can be off by 0.05-0.1 Å. Use at least def2-SVP or 6-31G*.

### ⚠️ 4. Not Verifying a Minimum
After optimization, **always** run a frequency calculation. If you find imaginary
frequencies, the geometry is a saddle point, not a minimum. Perturb along the
imaginary mode and re-optimize.

### ⚠️ 5. Symmetry Artifacts
Breaking symmetry artificially can lead to wrong structures. 
In ORCA, use `%geom UseSymm false end` if symmetry is causing issues.

### ⚠️ 6. Convergence Criteria Too Loose
Default thresholds are often fine, but for thermochemistry or reaction energies,
use `Convergence tight` to ensure accurate Hessians and frequencies.

## 🔬 Research Connection

Geometry optimization is the foundation of computational chemistry workflows:

- **Drug design**: Optimized protein-ligand binding geometries reveal key interactions
  (hydrogen bonds, π-stacking). Wrong geometry → wrong binding energy.
- **Transition metal catalysis**: Finding the correct geometry of metal-ligand complexes
  is critical for predicting reactivity. Often requires careful choice of spin state and functional.
- **Reaction mechanism**: Each point on the reaction pathway (reactant → TS → product)
  requires a geometry optimization to characterize.
- **Crystal structure prediction**: Modern approaches optimize thousands of candidate
  crystal structures at the DFT-D level.

**Notable achievement**: The 2022 Nobel Prize in Chemistry was awarded in part for
work that relied heavily on DFT geometry optimizations to design new click chemistry reactions.

## 📋 Summary

| Concept | Key Points |
|---------|----------|
| PES | $E(\mathbf{R})$: energy as function of nuclear coordinates |
| Minimum | All Hessian eigenvalues > 0; $\mathbf{g} = 0$ |
| Transition state | Exactly one negative Hessian eigenvalue |
| BFGS optimizer | Quasi-Newton; updates approximate inverse Hessian |
| Convergence | Check gradient, displacement, AND energy change |
| B3LYP/def2-SVP | Typical bond length error: ~5-15 mÅ vs experiment |
| Always verify | Run frequency calculation after optimization! |

**Typical B3LYP/def2-SVP performance:**
- Bond lengths: ±0.01 Å (1-15 mÅ error)
- Bond angles: ±0.5-2°
- Dihedral angles: ±2-5°
- Relative energies (conformers): ±1-3 kcal/mol

## 📝 Exercises

1. **CO₂ optimization**: Optimize CO₂ at B3LYP/def2-SVP. 
   - Does it remain linear? (Check the C-O-C angle)
   - What is the C=O bond length vs experimental 1.162 Å?
   - Hint: use `mol.atom = 'C 0 0 0; O 0 0 1.2; O 0 0 -1.2'`

2. **Methanol conformers**: Optimize methanol (CH₃OH) starting from both 
   staggered and eclipsed conformations (change the H-O-C-H dihedral angle).
   Do both converge to the same structure?

3. **Convergence criteria**: Optimize water with `geometric_solver.optimize(mf, verbose=0)`
   using default settings vs. Try running with a tighter tolerance by exploring 
   geometric_solver options. How many steps does each take?

4. **Method comparison**: Optimize the H-F bond length at RHF/6-31G* and B3LYP/6-31G*.
   Compare both with the experimental value of 0.9168 Å. Which method is more accurate?

5. **ORCA input writing**: Write a complete ORCA input file to optimize ethanol (C₂H₅OH)
   at the PBE0-D3(BJ)/def2-SVP level of theory. Include the `%pal` block for 
   4 processors and use `RIJCOSX` for efficiency.